# Unidad II · Preparación, ETL y Data Mart
Ulises González Coronado · Recursamiento ECBD 2026

Dataset Energy Efficiency de UCI. Ejecutar de arriba hacia abajo con el entorno del repositorio.

In [1]:
from pathlib import Path
import sys, json
import pandas as pd
from IPython.display import display, Image
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src/preparacion_datos.py').exists())
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
print('Proyecto:', ROOT.name)


Proyecto: REC_ECBD_2026_Gonzalez_Ulises


## 1. Extracción y copia de trabajo
Se conserva el original; el hash permite comprobarlo.

In [2]:
import hashlib
raw = ROOT / 'data/raw/ENB2012_data.xlsx'
firma = hashlib.sha256(raw.read_bytes()).hexdigest()
original = pd.read_excel(raw)
trabajo = original.copy(deep=True)
display(trabajo.head())

,X1,X2,X3,X4,X5,X6,X7,X8,Y1,Y2
0,0.98,514.5,294.0,110.25,7.0,2,0.0,0,15.55,21.33
1,0.98,514.5,294.0,110.25,7.0,3,0.0,0,15.55,21.33
2,0.98,514.5,294.0,110.25,7.0,4,0.0,0,15.55,21.33
3,0.98,514.5,294.0,110.25,7.0,5,0.0,0,15.55,21.33
4,0.90,563.5,318.5,122.50,7.0,2,0.0,0,20.84,28.28


## 2. Calidad y transformación justificada
No hay faltantes ni duplicados, por eso no se imputan ni eliminan filas. X6/X8 son códigos nominales. Los atípicos estadísticos no se eliminan sin evidencia de error. Se agregan identificadores, grupo de geometría y una clase didáctica.

In [3]:
from src.preparacion_datos import preparar
procesado, calidad = preparar()
display(pd.Series(calidad))
pd.testing.assert_frame_equal(original.astype(float), procesado[original.columns].astype(float))
assert hashlib.sha256(raw.read_bytes()).hexdigest() == firma
print('Los 768 registros originales se preservan exactamente.')

Preparación terminada: 768 registros, 8 entradas y 2 objetivos.
Faltantes: 0. Duplicados: 0. No se eliminaron ni imputaron datos.
Compacidad y altura: no existen compacidades compartidas por ambas alturas.
SQLite complementario validado. Para el modelo estrella en SQL Server: python -m src.etl_sqlserver --cargar


filas_originales                                                                768
filas_procesadas                                                                768
columnas_originales                                                              10
entradas                                                                          8
objetivos                                                                         2
faltantes                                                                         0
duplicados_completos                                                              0
duplicados_entradas                                                               0
filas_eliminadas                                                                  0
valores_imputados                                                                 0
geometrias                                                                       12
sha256_original                   0089fcffc1415e41e2ff63730cca5280efe54fc43d

Los 768 registros originales se preservan exactamente.


## 3. Modelo estrella y grano
Una fila de HECHOS representa una simulación. Altura forma parte de DimGeometria; proporción y distribución de cristal forman parte de DimAcristalamiento. Todas las dimensiones se conectan directamente a HECHOS. Las cargas se resumen por promedio, no por suma.

El modelo integrado incorpora resultados de UIII/UIV ya generados en el repositorio. Para reconstruir desde cero ejecutar primero `python -m src.ejecutar_proyecto`.

In [4]:
from src.etl_sqlserver import generar, RELACIONES
tablas = generar()
display(pd.DataFrame([{'tabla': n, 'filas': len(t)} for n,t in tablas.items()]))
display(pd.DataFrame(RELACIONES, columns=['hija','FK','padre','PK']))
display(tablas['HECHOS'].head())

,tabla,filas
0,DimGeometria,12
1,DimOrientacion,4
2,DimAcristalamiento,16
3,DimClase,2
4,DimConjunto,2
5,DimCluster,2
6,HECHOS,768
7,MetricasRegresion,4
8,MetricasClasificacion,2
9,EvaluacionClusters,7


,hija,FK,padre,PK
0,HECHOS,id_geometria,DimGeometria,id_geometria
1,HECHOS,id_orientacion,DimOrientacion,id_orientacion
2,HECHOS,id_acristalamiento,DimAcristalamiento,id_acristalamiento
3,HECHOS,id_clase,DimClase,id_clase
4,HECHOS,id_conjunto,DimConjunto,id_conjunto
5,HECHOS,id_cluster,DimCluster,id_cluster


,id_registro,id_geometria,id_orientacion,id_acristalamiento,id_clase,id_conjunto,id_cluster,Y1,Y2,Y1_predicha,Y2_predicha,clase_predicha,error_absoluto_Y1,error_absoluto_Y2,PC1,PC2
0,1,12,2,1,1,1,2,15.55,21.33,NaN,NaN,NaN,NaN,NaN,-3.128954,-0.902373
1,2,12,3,1,1,1,2,15.55,21.33,NaN,NaN,NaN,NaN,NaN,-3.128954,-0.902373
2,3,12,4,1,1,1,2,15.55,21.33,NaN,NaN,NaN,NaN,NaN,-3.128954,-0.902373
3,4,12,5,1,1,1,2,15.55,21.33,NaN,NaN,NaN,NaN,NaN,-3.128954,-0.902373
4,5,11,2,1,1,2,2,20.84,28.28,16.360067,22.416467,baja,4.479933,5.863533,-2.355562,-0.141667


## 4. Carga real a SQL Server
Requiere SQL Server local, sqlcmd y autenticación Windows. Solo reconstruye las tablas de la base dedicada REC_ECBD_Energia. La carga es transaccional y repetible.

In [5]:
from src.etl_sqlserver import cargar
verificacion = cargar('localhost')
print(verificacion['verificacion'])

Changed database context to 'master'.
Changed database context to 'REC_ECBD_Energia'.
tabla|filas
-----|-----
HECHOS|768
tabla|filas
-----|-----
DatasetOriginal|768
control|valor
-------|-----
Llaves foraneas activas|6
control|valor
-------|-----
Predicciones de prueba|192
calefaccion_media|refrigeracion_media
-----------------|-------------------
22.307195312499971|24.587760416666711
resultado
---------
RECONSTRUCCION_EXACTA_OK

Changed database context to 'master'.
Changed database context to 'REC_ECBD_Energia'.
tabla|filas
-----|-----
HECHOS|768
tabla|filas
-----|-----
DatasetOriginal|768
control|valor
-------|-----
Llaves foraneas activas|6
control|valor
-------|-----
Predicciones de prueba|192
calefaccion_media|refrigeracion_media
-----------------|-------------------
22.307195312499971|24.587760416666711
resultado
---------
RECONSTRUCCION_EXACTA_OK


## 5. Conclusión
La reconstrucción desde las dimensiones y HECHOS coincide con DatasetOriginal. Las seis llaves foráneas de HECHOS permanecen activas. Las predicciones están disponibles únicamente en los 192 casos de prueba; los nulos en entrenamiento son deliberados y no son errores de calidad.